# RAGAS Testset Generator

Generate question/answer/context triples from Vietnamese company policy documents  
using RAGAS 0.2.x `TestsetGenerator` + Gemini 2.5 Flash.

**Documents**: 6 HR policy `.txt` files in `data/`  
**Output**: `testset.csv` with columns `question`, `contexts`, `ground_truth`, `evolution_type`

## 1. Install Dependencies

Pin `ragas` to `0.2.x` — the `TestsetGenerator` / `generate_with_langchain_docs` API lives there.  
**Restart the kernel after running this cell** if a different ragas version was previously loaded.

In [1]:
%pip install -q \
    "ragas>=0.2.0,<0.3.0" \
    "langchain-google-genai>=1.0.0" \
    "langchain-community>=0.2.0,<0.4.0" \
    "python-dotenv>=1.0.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Environment Setup

LangChain Google reads `GOOGLE_API_KEY`, not `GEMINI_API_KEY`.  
This cell maps the project-standard variable automatically.

Create a `.env` file in this directory with:
```
GEMINI_API_KEY=your-key-here
```

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path(".env"), override=False)

gemini_key = os.getenv("GEMINI_API_KEY")
if gemini_key and not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = gemini_key
    print("Mapped GEMINI_API_KEY → GOOGLE_API_KEY")
elif os.getenv("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY already set")
else:
    raise EnvironmentError(
        "Neither GEMINI_API_KEY nor GOOGLE_API_KEY is set. "
        "Create a .env file in this directory with GEMINI_API_KEY=<your-key>"
    )

print(f"API key length: {len(os.environ['GOOGLE_API_KEY'])} chars")

GOOGLE_API_KEY already set
API key length: 53 chars


## 3. Load Documents

Load all `.txt` files from `data/` with UTF-8 encoding (required for Vietnamese text).

In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)

documents = loader.load()

print(f"\nLoaded {len(documents)} documents")
print(f"{'File':<45} {'Chars':>8}")
print("-" * 55)
for doc in sorted(documents, key=lambda d: d.metadata.get("source", "")):
    fname = Path(doc.metadata.get("source", "?")).name
    print(f"  {fname:<43} {len(doc.page_content):>8,}")

100%|██████████| 6/6 [00:00<00:00, 2289.88it/s]


Loaded 6 documents
File                                             Chars
-------------------------------------------------------
  chinh_sach_lam_viec_tu_xa.txt                  4,718
  chinh_sach_nghi_phep.txt                       4,140
  chinh_sach_phuc_loi.txt                        5,122
  quy_dinh_trang_phuc.txt                        4,566
  quy_trinh_xin_phep.txt                         4,690
  so_tay_nhan_vien.txt                           5,824


## 4. Initialize LLM and Embeddings

- **LLM**: `gemini-2.5-flash` via `ChatGoogleGenerativeAI`
- **Embeddings**: `models/text-embedding-004` — the `models/` prefix is required by `langchain-google-genai`

Both are wrapped with RAGAS's `LangchainLLMWrapper` / `LangchainEmbeddingsWrapper`.

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
)

# models/ prefix is required for GoogleGenerativeAIEmbeddings
_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
)

generator_llm = LangchainLLMWrapper(_llm)
generator_embeddings = LangchainEmbeddingsWrapper(_embeddings)

print("LLM:", _llm.model)
print("Embeddings:", _embeddings.model)
print("Wrappers ready.")

/opt/anaconda3/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


LLM: models/gemini-2.5-flash
Embeddings: models/gemini-embedding-001
Wrappers ready.


## 5. Generate Testset

Runs `TestsetGenerator.generate_with_langchain_docs()` with `testset_size=10`.

Internally RAGAS:
1. Chunks the documents and builds an embedding-based knowledge graph
2. Synthesizes questions of mixed types (`simple`, `reasoning`, `multi_context`)
3. Generates ground-truth answers via Gemini

**Allow 2–5 minutes** for the Gemini API calls to complete.

In [ ]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,  # note: param name is embedding_model, not embeddings
)

testset = generator.generate_with_langchain_docs(
    documents,
    testset_size=20,
)

print(f"\nGenerated {len(testset)} samples.")

## 6. Inspect the Generated Dataset

In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 20)

df = testset.to_pandas()

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nQuestion types:\n{df['evolution_type'].value_counts().to_string()}\n")
df

## 7. Save to CSV

`contexts` is a list — JSON-encode it so it survives CSV round-tripping.  
`utf-8-sig` encoding adds a BOM so Excel opens Vietnamese text correctly.

In [ ]:
import json

OUTPUT_PATH = Path("testset.csv")

df_save = df.copy()
if "contexts" in df_save.columns:
    df_save["contexts"] = df_save["contexts"].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
    )

df_save.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved {len(df_save)} rows → {OUTPUT_PATH.resolve()}")
print(f"File size: {OUTPUT_PATH.stat().st_size:,} bytes")
print("\nTo reload:")
print("  df = pd.read_csv('testset.csv')")
print("  df['contexts'] = df['contexts'].apply(json.loads)")